In [2]:
from calendar import monthrange
import sqlite3
import pandas as pd
import datetime
from dateutil.relativedelta import relativedelta
import os


## Check Inventory

In [3]:
conn = sqlite3.connect('Data/DB/allData.db')

In [30]:
df = pd.read_sql("select * from data where shortName='TMP' and level='2 m above ground'",conn)


In [31]:
df.shape

(45268, 45)

In [4]:
df = pd.read_sql("select * from data",conn)

In [5]:
df.shape

(497948, 45)

In [6]:
df.columns

Index(['index', 'file', 'refDate', 'typeOfData', 'fullName', 'shortName',
       'paramNum', 'units', 'validDate', 'forecastDate', 'valueOfForecastTime',
       'unitOfForecastTime', 'level', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7',
       'p8', 'p9', 'p10', 'p11', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17',
       'p18', 'p19', 'p20', 'p21', 'p22', 'p23', 'p24', 'p25', 'p26', 'p27',
       'p28', 'p29', 'p30', 'p31', 'p32'],
      dtype='object')

In [7]:
dfI = df[["fullName","shortName","level","validDate","forecastDate","valueOfForecastTime"]]

In [8]:
dfI["fullName"].value_counts()

fullName
Temperature                         90536
Volumetric Soil Moisture Content    90536
Pressure                            45268
Potential Evaporation Rate          45268
Maximum Temperature                 45268
Minimum Temperature                 45268
Specific Humidity                   45268
Maximum specific humidity at 2m     45268
Minimum specific humidity at 2m     45268
Name: count, dtype: int64

In [27]:
dfI.shape

(497948, 5)

In [12]:
dfI["validDate"] = pd.to_datetime(dfI["validDate"])
dfI["forecastDate"] = pd.to_datetime(dfI["forecastDate"])
dfI["forecastDate"] = dfI["forecastDate"].dt.date
dfI["validDate"] = dfI["validDate"].dt.date

dfI["forecastDate"] = dfI["forecastDate"].astype(str)
dfI["validDate"] = dfI["validDate"].astype(str)

/tmp/ipykernel_10443/3786243449.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfI["validDate"] = pd.to_datetime(dfI["validDate"])
/tmp/ipykernel_10443/3786243449.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfI["forecastDate"] = pd.to_datetime(dfI["forecastDate"])
/tmp/ipykernel_10443/3786243449.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://p

In [17]:
invt = {}
invt_yrmo = {}
for index,row in dfI.iterrows():
    var = row["shortName"]
    lev = row["level"]
    date = str(row["validDate"])
    fct  = str(row["forecastDate"])
    fdel = row["valueOfForecastTime"]
    if var not in invt:
        invt[var]={}
        
    if lev not in invt[var]:
        invt[var][lev]={}

    if date not in invt[var][lev]:
        invt[var][lev][date]={}
        
    if fdel not in invt[var][lev][date]:
        invt[var][lev][date][fdel]=0
    
    invt[var][lev][date][fdel]+=1
    year = row["validDate"][0:4]
    month = row["validDate"][5:7]

    if var not in invt_yrmo:
        invt_yrmo[var]={}
    if lev not in invt_yrmo[var]:
        invt_yrmo[var][lev]={}
        
    if year not in invt_yrmo[var][lev]:
        invt_yrmo[var][lev][year]={}
        
    if month not in invt_yrmo[var][lev][year]:
        invt_yrmo[var][lev][year][month]=0
        
    invt_yrmo[var][lev][year][month]+=1

In [21]:
for var,dct in invt_yrmo.items():
    for lev,dct2 in dct.items():
        for yr,inv in sorted(dct2.items()):
            print(f"{var:5.5s} {lev:20.20s} {yr:4s}",end="")
            for mo in ["01","02","03","04","05","06","07","08","09","10","11","12"]:
                if mo in inv:
                    cnt=inv[mo]
                else:
                    cnt=0
                print(f" {cnt:4d}",end="")
            print("")
        print("------")


PRES  surface              2011    0    0    0  294  303  185  303  294  294  303  294  303
PRES  surface              2012  303  285  293  294  303  294    0  283  294  293  285  303
PRES  surface              2013  303  276  303  294  293  274  303  303  294  303  294  301
PRES  surface              2014  303  276  303  294  303  294  303  302  294  303  294  303
PRES  surface              2015  303  276  303  294  303  294  303  303  294  303  294  303
PRES  surface              2016  310  290  310  300  310  300  310  310  300  310  300  300
PRES  surface              2017  310  280  310  300  310  300  310  310  300  310  300  310
PRES  surface              2018  310  280  310  300  310  300  310  310  300  310  300  310
PRES  surface              2019  310  280  310  300  301  299  310  310  300  303  300  310
PRES  surface              2020  310  290  310  300  310  300  310  310  300  310  299  310
PRES  surface              2021  310  280  250  300  290  278  310  310  300  31

In [ ]:
invt

In [13]:
syear=2016
eyear = 2018
dates2Check = []
for year in range(syear,eyear+1):
    for month in range(1,13):
        _,ndays = monthrange(year,month)
        for day in range(1,ndays+1):
            mon = str(month).zfill(2)
            dy  = str(day).zfill(2)
            ky = f"{year}-{mon}-{dy}" 
            dates2Check.append(ky)

In [14]:

missed = []
check={}
for var in invt.keys():
    for lev in invt[var]:
        for date in dates2Check:
            if date in invt[var][lev]:
                if var not in check:
                    check[var]={}
                if lev not in check[var]:
                    check[var][lev]={}
                if date not in check[var][lev]:
                    check[var][lev][date]={}
                    check[var][lev][date]['good']=0
                    check[var][lev][date]['bad']={}
                    
                for nn in range(0,10):
                    if nn in invt[var][lev][date]:
                        if invt[var][lev][date][nn]==1:
                            check[var][lev][date]['good']+=1
                        else:
                            check[var][lev][date]['bad'][nn]=invt[var][lev][date][nn]
                    else:
                        check[var][lev][date]['bad'][nn] = "MISS"
                            
            else:
               missed.append(date)
               

In [15]:
summary={}
for var in check.keys():
    for lev in check[var].keys():
        for date in check[var][lev].keys():
            year = date[0:4]
            if check[var][lev][date]['good'] == 10:
                
                if var not in summary:
                    summary[var]={}
                if lev not in summary[var]:
                    summary[var][lev]={}
                if year not in summary[var][lev]:
                    summary[var][lev][year]=0
                    
                summary[var][lev][year]+=1   
            else:
                print("BAD BAD ",var,lev,date,check[var][lev][date]['good'])

In [16]:
summary

{'PRES': {'surface': {'2016': 365, '2017': 365, '2018': 365}},
 'TMP': {'surface': {'2016': 365, '2017': 365, '2018': 365},
  '2 m above ground': {'2016': 365, '2017': 365, '2018': 365}},
 'SOILW': {'0-0.1 m underground': {'2016': 365, '2017': 365, '2018': 365},
  '0.1-0.4 m underground': {'2016': 365, '2017': 365, '2018': 365}},
 'PEVPR': {'surface': {'2016': 365, '2017': 365, '2018': 365}},
 'TMAX': {'2 m above ground': {'2016': 365, '2017': 365, '2018': 365}},
 'TMIN': {'2 m above ground': {'2016': 365, '2017': 365, '2018': 365}},
 'SPFH': {'2 m above ground': {'2016': 365, '2017': 365, '2018': 365}},
 'QMAX': {'2 m above ground': {'2016': 365, '2017': 365, '2018': 365}},
 'QMIN': {'2 m above ground': {'2016': 365, '2017': 365, '2018': 365}}}

In [ ]:
missed

## Check Files

In [6]:
yearT = 2019
month = 1
mon = str(month).zfill(2)
day = 1
dy = str(day).zfill(2)
files=[]
badfiles=[]
for month in range(1,13):
        mon = str(month).zfill(2)
        day = 1
        dy = str(day).zfill(2)
        tdate = datetime.date(yearT,month,int(dy))
        _,ndays = monthrange(tdate.year,tdate.month)
        for nd in range(0,ndays):
            adate = tdate + relativedelta(days=nd)
            year = adate.year
            month = str(adate.month).zfill(2)
            day = str(adate.day).zfill(2)
        #    print(year,month,day)
            for fp in range(0,10):
                fdate = adate + relativedelta(months=fp)
                yearfc = fdate.year
                monthfc = fdate.month
                monfc = str(monthfc).zfill(2)
                file=f"flxf.01.{year}{mon}{day}00.{yearfc}{monfc}.avrg.csv"
                if not os.path.exists(f"Data/TextFiles/{yearT}/{file}"):
                   print("NOT FOUND ",file)
                else:
                   files.append(file)

NOT FOUND  flxf.01.2019052900.201905.avrg.csv
NOT FOUND  flxf.01.2019052900.201906.avrg.csv
NOT FOUND  flxf.01.2019052900.201907.avrg.csv
NOT FOUND  flxf.01.2019052900.201908.avrg.csv
NOT FOUND  flxf.01.2019053000.201911.avrg.csv
NOT FOUND  flxf.01.2019053100.201905.avrg.csv
NOT FOUND  flxf.01.2019053100.201906.avrg.csv
NOT FOUND  flxf.01.2019053100.201907.avrg.csv
NOT FOUND  flxf.01.2019053100.201908.avrg.csv
NOT FOUND  flxf.01.2019061300.201909.avrg.csv
NOT FOUND  flxf.01.2019102100.201910.avrg.csv
NOT FOUND  flxf.01.2019102100.201911.avrg.csv
NOT FOUND  flxf.01.2019102100.201912.avrg.csv
NOT FOUND  flxf.01.2019102100.202001.avrg.csv
NOT FOUND  flxf.01.2019102100.202002.avrg.csv
NOT FOUND  flxf.01.2019102100.202003.avrg.csv
NOT FOUND  flxf.01.2019102100.202004.avrg.csv
